### Common rules for all problems

- Use Python and implement the specified class. Do not change public method names, parameters, or return formats.
- Use only the standard library. No databases, network calls, or external services.
- All state is in memory, and execution is single-threaded.
- Examples show method calls and their return values; no console-input parser is required.
- Unless specified otherwise, mutating methods return None.
- Invalid input must not partially change state.
- Returned mutable collections must not let callers modify internal state.
- You may introduce private methods and helper classes.
- Treat each problem independently.

1. Chat Conversation History Manager

Difficulty: Easy · Target: 20–25 minutes
Concepts: Classes, lists, dictionaries, validation, defensive copying.

Problem statement:
Store messages in the exact order they are added. Each message has exactly two attributes: role and content. Roles must be "user" or "assistant". Consecutive messages from the same role are allowed.

Required interface:

In [ ]:
class ChatHistory:
    def add_message(self, role: str, content: str) -> None: ...
    def get_history(self) -> list[dict[str, str]]: ...

# Input and expected output:

chat = ChatHistory()
chat.add_message("user", "Hello")
chat.add_message("assistant", "Hi!")

chat.get_history()
# [
#     {"role": "user", "content": "Hello"},
#     {"role": "assistant", "content": "Hi!"}
# ]

Constraints and traps:

- Empty content is valid.
- Invalid roles raise ValueError; non-string content raises TypeError.
- Empty history returns [].
- Changing a dictionary inside returned history must not affect stored messages.
- Adding a message should take amortized O(1) time.
Optional hint: A copy of the outer list alone does not isolate the dictionaries inside it.

2. Bounded Conversation Buffer

Difficulty: Easy · Target: 20–25 minutes
Concepts: Bounded storage, queues, eviction.

Problem statement:
Store only the most recent capacity messages. When full, adding a message evicts the oldest one. Preserve the order of retained messages.

Required interface:

In [ ]:
class ConversationBuffer:
    def __init__(self, capacity: int): ...
    def add_message(self, role: str, content: str) -> None: ...
    def get_history(self) -> list[dict[str, str]]: ...

# Input and expected output

buffer = ConversationBuffer(2)
buffer.add_message("user", "A")
buffer.add_message("assistant", "B")
buffer.add_message("user", "C")

buffer.get_history()
# [
#     {"role": "assistant", "content": "B"},
#     {"role": "user", "content": "C"}
# ]


Constraints and traps:
- capacity >= 0; negative capacity raises ValueError.
- Capacity zero accepts valid messages but retains nothing.
- Apply Problem 1’s message validation.
- Addition and eviction must be O(1).
- Do not use list.pop(0).
Optional hint: Consider a deque or a circular buffer.

3. Multi-Conversation Registry

Difficulty: Easy–Medium · Target: 25–30 minutes
Concepts: Nested collections, object ownership, isolation.

Problem statement:
Manage multiple independent conversations identified by string IDs.

Required interface:


In [ ]:
class ConversationRegistry:
    def create(self, conversation_id: str) -> bool: ...
    def add_message(
        self, conversation_id: str, role: str, content: str
    ) -> None: ...
    def get_history(self, conversation_id: str) -> list[dict[str, str]]: ...
    def delete(self, conversation_id: str) -> bool: ...

# Input and expected output
registry = ConversationRegistry()
registry.create("c1")                    # True
registry.create("c1")                    # False
registry.create("c2")                    # True
registry.add_message("c1", "user", "Hi")

registry.get_history("c2")               # []
registry.delete("c1")                    # True
registry.delete("c1")                    # False

Constraints and traps:
- Adding to or reading an unknown conversation raises KeyError.
- Duplicate creation must not erase history.
- Conversation IDs must be nonempty.
- Recreating a deleted ID starts an empty conversation.
- Apply Problem 1’s message validation.
Optional hint: Watch for accidentally sharing the same list across conversations.

4. Idempotent Message Inbox

Difficulty: Easy–Medium · Target: 25–30 minutes
Concepts: Dictionaries, deduplication, consistency.

Problem statement:
Messages arrive with IDs and may be retried. Store each message once. An exact retry is harmless, but reusing an ID with different content is an error.

Required interface:

In [ ]:
class MessageInbox:
    def receive(self, message_id: str, content: str) -> bool: ...
    def get_messages(self) -> list[tuple[str, str]]: ...

# Input and expected output 
inbox = MessageInbox()
inbox.receive("m1", "Hello")              # True
inbox.receive("m1", "Hello")              # False
inbox.receive("m2", "World")              # True

inbox.get_messages()
# [("m1", "Hello"), ("m2", "World")]

inbox.receive("m1", "Changed")
# Raises ValueError

Constraints and traps:

- IDs are nonempty; empty content is valid.
- Preserve first-arrival order.
- Duplicate retries must not change ordering.
- ID lookup should take average O(1) time.
- Assume Python 3.7 or later.

Optional hint: Membership and content equality answer different questions.

5. Conversation Browser History

Difficulty: Easy–Medium · Target: 25–30 minutes
Concepts: Lists, stacks, cursor state.

Problem statement:
Track the conversations a user navigates through. Support moving backward and forward. Visiting a new conversation after moving backward discards forward history.

Required interface:

In [ ]:
class ConversationBrowser:
    def __init__(self, initial_id: str): ...
    def visit(self, conversation_id: str) -> None: ...
    def back(self, steps: int) -> str: ...
    def forward(self, steps: int) -> str: ...
    def current(self) -> str: ...

# Input and expected output

browser = ConversationBrowser("A")
browser.visit("B")
browser.visit("C")
browser.back(1)                          # "B"
browser.visit("D")
browser.forward(10)                      # "D"
browser.back(10)                         # "A"


Constraints and traps:
- Negative steps raise ValueError; zero leaves the position unchanged.
- Excessive steps stop at the relevant boundary.
- Repeated visits to the same ID are separate history entries.
- Do not confuse navigation history with a set of unique conversations.
Optional hint: One list and a cursor are enough.

6. Draft Editor with Undo and Redo

Difficulty: Medium · Target: 30–35 minutes
Concepts: Stacks, reversible operations, state transitions.

Problem statement:
Maintain a draft string. Support appending text, deleting characters from the end, and undoing or redoing successful edits.

Required interface:

In [ ]:
class DraftEditor:
    def append(self, text: str) -> None: ...
    def delete(self, count: int) -> None: ...
    def undo(self) -> bool: ...
    def redo(self) -> bool: ...
    def get_text(self) -> str: ...

# Input and Expected Output
editor = DraftEditor()
editor.append("Hello")
editor.append(" AI")
editor.delete(3)
editor.get_text()                        # "Hello"
editor.undo()                            # True
editor.get_text()                        # "Hello AI"
editor.undo()                            # True
editor.get_text()                        # "Hello"
editor.redo()                            # True

Constraints and traps:
- Deleting more characters than exist deletes everything.
- Negative counts raise ValueError.
- No-op edits create no undo record and preserve redo history.
- A successful new edit clears redo history.
- Empty undo or redo returns False.
Optional hint: Start with snapshots; then discuss storing only the data needed to reverse each edit.